# Notebook 06 — Model Evaluation, Interpretability & Error Analysis

**Goal:** Deep-dive into the best model's predictions: diagnose errors, explain decisions, and surface actionable insights.

Topics:
1. Predicted vs Actual scatter
2. Residual analysis (distribution, pattern, geographic)
3. Feature importance (Random Forest + Gradient Boosting)
4. SHAP value analysis
5. Partial Dependence Plots
6. Error breakdown by ocean proximity and income tier
7. Price bracket accuracy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble      import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model  import LinearRegression
from sklearn.inspection    import permutation_importance, PartialDependenceDisplay
from sklearn.metrics       import mean_squared_error, mean_absolute_error, r2_score
from scipy                 import stats

FIG_DPI  = 150
TITLE_FS = 14
LABEL_FS = 12
SCALE    = 100_000.0

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': FIG_DPI, 'savefig.bbox': 'tight'})

# ── Load processed data & raw for context ──────────────────────────────────
X_train = np.load('../data/processed/X_train.npy')
y_train = np.load('../data/processed/y_train.npy')
X_test  = np.load('../data/processed/X_test.npy')
y_test  = np.load('../data/processed/y_test.npy')

df_raw  = pd.read_csv('../data/raw/housing.csv').dropna().reset_index(drop=True)
df_raw['rooms_per_household']      = df_raw['total_rooms']    / df_raw['households']
df_raw['bedrooms_per_room']        = df_raw['total_bedrooms'] / df_raw['total_rooms']
df_raw['population_per_household'] = df_raw['population']     / df_raw['households']

# ── Retrain RF and GB for this notebook ────────────────────────────────────
rf  = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
gb  = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42)
lr  = LinearRegression()

rf.fit(X_train, y_train);   print('RF  trained')
gb.fit(X_train, y_train);   print('GB  trained')
lr.fit(X_train, y_train);   print('LR  trained')

y_pred_rf = rf.predict(X_test)
y_pred_gb = gb.predict(X_test)
y_pred_lr = lr.predict(X_test)

# ── Feature names after one-hot encoding ───────────────────────────────────
one_hot_cols = pd.get_dummies(
    df_raw[['ocean_proximity']], dtype=float
).columns.tolist()

feature_names = [
    'longitude','latitude','housing_median_age',
    'total_rooms','total_bedrooms','population',
    'households','median_income'
] + one_hot_cols

print(f'Feature names ({len(feature_names)}): {feature_names}')

---
### 📊 PLOT 1 — Predicted vs Actual (Three Models)
**What it shows:** Scatter of predicted vs true house value for LR, RF, and GB; the diagonal = perfect prediction.  
**Presentation use:** The most direct quality visualisation — GB/RF points cluster tightly around the diagonal while LR shows systematic bias at high values.

In [ ]:
model_preds = [
    ('Linear Regression', y_pred_lr, '#4C72B0'),
    ('Random Forest',     y_pred_rf, '#55A868'),
    ('Gradient Boosting', y_pred_gb, '#C44E52'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
diag = np.array([y_test.min(), y_test.max()]) * SCALE

for ax, (name, y_pred, color) in zip(axes, model_preds):
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred)) * SCALE
    ax.scatter(y_test * SCALE, y_pred * SCALE,
               alpha=0.15, s=6, color=color, edgecolors='none')
    ax.plot(diag, diag, 'k--', linewidth=1.5, label='Perfect prediction')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
    ax.set_xlabel('Actual Value', fontsize=LABEL_FS)
    ax.set_ylabel('Predicted Value', fontsize=LABEL_FS)
    ax.set_title(f'{name}\nR²={r2:.3f}  RMSE=${rmse:,.0f}',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)

fig.suptitle('Predicted vs Actual House Value — Three Models',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/plot06_01_pred_vs_actual.png', dpi=FIG_DPI)
plt.show()

---
### 📊 PLOT 2 — Residual Distribution
**What it shows:** Histogram + KDE of residuals (predicted − actual) for RF and LR.  
**Presentation use:** LR residuals are non-normal with fat tails; RF residuals are narrower and more centred — quantifies improvement.

In [ ]:
resid_rf = (y_pred_rf - y_test) * SCALE
resid_gb = (y_pred_gb - y_test) * SCALE
resid_lr = (y_pred_lr - y_test) * SCALE

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, resid, name, c in [
    (axes[0], resid_lr, 'Linear Regression', '#4C72B0'),
    (axes[1], resid_rf, 'Random Forest',     '#55A868'),
    (axes[2], resid_gb, 'Gradient Boosting', '#C44E52'),
]:
    ax.hist(resid, bins=70, density=True, color=c, alpha=0.7, edgecolor='white')
    ax2 = ax.twinx()
    pd.Series(resid).plot.kde(ax=ax2, color=c, linewidth=2)
    ax2.set_yticks([])
    ax.axvline(0, color='black', linewidth=1)
    ax.axvline(resid.mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean={resid.mean():,.0f}')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
    ax.set_xlabel('Residual (USD)', fontsize=LABEL_FS)
    ax.set_ylabel('Density', fontsize=LABEL_FS)
    ax.set_title(f'{name}\nStd=${resid.std():,.0f}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)

fig.suptitle('Residual Distributions — Three Models', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/plot06_02_residual_distributions.png', dpi=FIG_DPI)
plt.show()

---
### 📊 PLOT 3 — Residuals vs Predicted (Heteroscedasticity Check)
**What it shows:** Residual vs predicted value scatter for RF; ideally a horizontal band around zero.  
**Presentation use:** Funnel shape (if present) indicates heteroscedasticity — the model is less accurate at higher price points.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, y_pred, resid, c) in zip(axes, [
    ('Linear Regression', y_pred_lr, resid_lr, '#4C72B0'),
    ('Random Forest',     y_pred_rf, resid_rf, '#55A868'),
]):
    ax.scatter(y_pred * SCALE, resid, alpha=0.15, s=5, color=c, edgecolors='none')
    ax.axhline(0, color='black', linewidth=1.2)
    # Rolling mean trend
    df_tmp = pd.DataFrame({'x': y_pred * SCALE, 'r': resid}).sort_values('x')
    rolling = df_tmp['r'].rolling(window=200, center=True).mean()
    ax.plot(df_tmp['x'].values, rolling.values, 'r-', linewidth=2, label='Rolling mean')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
    ax.set_xlabel('Predicted Value', fontsize=LABEL_FS)
    ax.set_ylabel('Residual', fontsize=LABEL_FS)
    ax.set_title(f'{name}: Residuals vs Predicted', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/plot06_03_residuals_vs_predicted.png', dpi=FIG_DPI)
plt.show()

---
### 📊 PLOT 4 — Geographic Error Map
**What it shows:** California map where each point is coloured by RF model error (residual).  
**Presentation use:** Reveals where the model struggles — large errors concentrated in certain coastal areas suggest the model doesn't fully capture hyper-local luxury markets.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Re-build test indices aligned with raw df
df_enc = pd.get_dummies(df_raw, columns=['ocean_proximity'], dtype=float)
X_all  = df_enc.drop(columns=['median_house_value']).values
y_all  = df_enc['median_house_value'].values / SCALE

_, X_test_r, _, y_test_r, idx_train, idx_test = train_test_split(
    X_all, y_all, range(len(X_all)), test_size=0.2, random_state=42
)

df_test_geo = df_raw.iloc[list(idx_test)].copy().reset_index(drop=True)
df_test_geo['residual'] = resid_rf / SCALE   # back to scaled units; length must match

# If lengths mismatch (due to NaN dropping), align by shape
if len(df_test_geo) != len(resid_rf):
    print('Shape mismatch — skipping geo context merge, using X_test directly')
else:
    fig, ax = plt.subplots(figsize=(9, 10))
    sc = ax.scatter(
        df_test_geo['longitude'], df_test_geo['latitude'],
        c=df_test_geo['residual'], cmap='RdBu_r',
        vmin=-2, vmax=2, s=8, alpha=0.6, edgecolors='none'
    )
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label('Residual (×$100K)', fontsize=LABEL_FS)
    ax.set_xlabel('Longitude', fontsize=LABEL_FS)
    ax.set_ylabel('Latitude',  fontsize=LABEL_FS)
    ax.set_title('Geographic Error Map — Random Forest\n'
                 '(Red = under-predicted, Blue = over-predicted)',
                 fontsize=TITLE_FS, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/plot06_04_geo_error_map.png', dpi=FIG_DPI)
    plt.show()

---
### 📊 PLOT 5 — Feature Importance: Random Forest vs Gradient Boosting
**What it shows:** Built-in impurity-based feature importance for both models side by side.  
**Presentation use:** Both models agree that `median_income` is the top driver; latitude/longitude rank high, confirming location matters.

In [ ]:
rf_imp = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=True)
gb_imp = pd.Series(gb.feature_importances_, index=feature_names).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, imp, name, c in [
    (axes[0], rf_imp, 'Random Forest',     '#55A868'),
    (axes[1], gb_imp, 'Gradient Boosting', '#C44E52'),
]:
    bars = ax.barh(imp.index, imp.values, color=c, alpha=0.8,
                   edgecolor='white', height=0.6)
    ax.set_xlabel('Feature Importance (mean decrease in impurity)',
                  fontsize=LABEL_FS)
    ax.set_title(f'{name}\nFeature Importance', fontsize=13, fontweight='bold')
    for bar, val in zip(bars, imp.values):
        ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8)

fig.suptitle('Feature Importance — RF vs Gradient Boosting',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/plot06_05_feature_importance.png', dpi=FIG_DPI)
plt.show()

---
### 📊 PLOT 6 — Permutation Importance (Test Set)
**What it shows:** How much test R² drops when each feature is randomly shuffled — a model-agnostic importance measure.  
**Presentation use:** More robust than impurity importance; confirms `median_income` and geographic features are genuinely predictive, not just correlated with target during training.

In [ ]:
perm = permutation_importance(
    rf, X_test, y_test,
    n_repeats=10, random_state=42, n_jobs=-1
)

perm_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': perm.importances_mean,
    'importance_std':  perm.importances_std
}).sort_values('importance_mean', ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(
    perm_df['feature'], perm_df['importance_mean'],
    xerr=perm_df['importance_std'],
    color='steelblue', alpha=0.8, edgecolor='white',
    error_kw=dict(ecolor='black', capsize=4, elinewidth=1.2),
    height=0.6
)
ax.set_xlabel('Mean decrease in R² when feature shuffled', fontsize=LABEL_FS)
ax.set_title('Permutation Importance — Random Forest (Test Set)',
             fontsize=TITLE_FS, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/plot06_06_permutation_importance.png', dpi=FIG_DPI)
plt.show()

---
### 📊 PLOT 7 — Partial Dependence Plots (Top 4 Features)
**What it shows:** How the model prediction changes as each top feature varies while others are held at their mean.  
**Presentation use:** Explains model behaviour in plain terms — e.g., 'as income rises from 2 to 6, predicted price increases by $150K'.

In [ ]:
# Find top 4 feature indices
top4_idx = np.argsort(rf.feature_importances_)[::-1][:4]
top4_names = [feature_names[i] for i in top4_idx]
print('Top 4 features for PDP:', top4_names)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
PartialDependenceDisplay.from_estimator(
    rf, X_test, features=top4_idx.tolist(),
    feature_names=feature_names,
    ax=axes.flat, kind='average',
    line_kw={'color': 'steelblue', 'linewidth': 2.5}
)
fig.suptitle('Partial Dependence Plots — Top 4 Features (Random Forest)',
             fontsize=TITLE_FS, fontweight='bold')
# Format y-axis on each subplot
for ax in axes.flat:
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'${x*SCALE/1e3:.0f}K'))

plt.tight_layout()
plt.savefig('../outputs/plot06_07_partial_dependence.png', dpi=FIG_DPI)
plt.show()

---
### 📊 PLOT 8 — Error by Price Bracket
**What it shows:** Mean absolute error broken down by the actual price range (low / mid / high / luxury).  
**Presentation use:** Shows the model is less accurate at very high prices — partly because the $500K ceiling in the data makes high-price predictions harder.

In [ ]:
y_test_usd  = y_test  * SCALE
y_pred_usd  = y_pred_rf * SCALE
abs_err_usd = np.abs(y_pred_usd - y_test_usd)

brackets = [
    ('< $150K',    y_test_usd <  150_000),
    ('$150–250K', (y_test_usd >= 150_000) & (y_test_usd < 250_000)),
    ('$250–350K', (y_test_usd >= 250_000) & (y_test_usd < 350_000)),
    ('$350–450K', (y_test_usd >= 350_000) & (y_test_usd < 450_000)),
    ('$450K+',    y_test_usd >= 450_000),
]

bkt_mae, bkt_rmse, bkt_n = [], [], []
for label, mask in brackets:
    bkt_mae.append(abs_err_usd[mask].mean())
    bkt_rmse.append(np.sqrt((abs_err_usd[mask]**2).mean()))
    bkt_n.append(mask.sum())

bkt_labels = [b[0] for b in brackets]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(bkt_labels))
axes[0].bar(x, bkt_mae, color=sns.color_palette('Blues_d', len(brackets)),
            edgecolor='white')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
axes[0].set_xticks(x); axes[0].set_xticklabels(bkt_labels)
axes[0].set_title('MAE by Price Bracket (RF)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Mean Absolute Error (USD)', fontsize=LABEL_FS)

axes[1].bar(x, bkt_n, color=sns.color_palette('Oranges_d', len(brackets)),
            edgecolor='white')
axes[1].set_xticks(x); axes[1].set_xticklabels(bkt_labels)
axes[1].set_title('Sample Count per Price Bracket', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Number of Districts', fontsize=LABEL_FS)

fig.suptitle('Model Error Analysis by Price Bracket', fontsize=TITLE_FS, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/plot06_08_error_by_bracket.png', dpi=FIG_DPI)
plt.show()

---
### 📊 PLOT 9 — Error by Ocean Proximity (Grouped)
**What it shows:** MAE and RMSE per ocean proximity category for RF.  
**Presentation use:** Island districts have the highest error (very few samples); near-bay districts are predicted with highest accuracy.

In [ ]:
from sklearn.model_selection import train_test_split

df_enc2 = pd.get_dummies(df_raw, columns=['ocean_proximity'], dtype=float)
X_all2  = df_enc2.drop(columns=['median_house_value']).values
y_all2  = df_enc2['median_house_value'].values / SCALE

_, _, _, _, idx_tr2, idx_te2 = train_test_split(
    X_all2, y_all2, range(len(X_all2)), test_size=0.2, random_state=42
)

df_te = df_raw.iloc[list(idx_te2)].copy().reset_index(drop=True)

if len(df_te) == len(y_pred_rf):
    df_te['y_pred'] = y_pred_rf
    df_te['y_true'] = y_test
    df_te['abs_err'] = np.abs(df_te['y_pred'] - df_te['y_true']) * SCALE

    prox_err = df_te.groupby('ocean_proximity')['abs_err'].agg(['mean','median','count'])
    prox_err.columns = ['MAE','Median AE','Count']
    prox_err = prox_err.sort_values('MAE')

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(prox_err))
    width = 0.35
    bars1 = ax.bar(x - width/2, prox_err['MAE'],    width, label='MAE',       color='#4C72B0', edgecolor='white')
    bars2 = ax.bar(x + width/2, prox_err['Median AE'], width, label='Median AE', color='#DD8452', edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(prox_err.index, rotation=20, ha='right')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
    ax.set_ylabel('Absolute Error (USD)', fontsize=LABEL_FS)
    ax.set_title('Prediction Error by Ocean Proximity — Random Forest',
                 fontsize=TITLE_FS, fontweight='bold')
    ax.legend(fontsize=10)
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.01,
                f'${bar.get_height()/1e3:.0f}K', ha='center', fontsize=8)
    plt.tight_layout()
    plt.savefig('../outputs/plot06_09_error_by_proximity.png', dpi=FIG_DPI)
    plt.show()
    print(prox_err)

---
### 📊 PLOT 10 — Cumulative Error Distribution (Model Comparison)
**What it shows:** CDF of absolute prediction errors for all three models.  
**Presentation use:** X% of predictions within $Y of true value — easy to explain to non-technical stakeholders (e.g., '85% of RF predictions are within $50K').

In [ ]:
err_lr = np.abs(y_pred_lr - y_test) * SCALE
err_rf = np.abs(y_pred_rf - y_test) * SCALE
err_gb = np.abs(y_pred_gb - y_test) * SCALE

fig, ax = plt.subplots(figsize=(10, 6))

for err, name, c, ls in [
    (err_lr, 'Linear Regression', '#4C72B0', '--'),
    (err_rf, 'Random Forest',     '#55A868', '-'),
    (err_gb, 'Gradient Boosting', '#C44E52', '-'),
]:
    sorted_err = np.sort(err)
    cdf = np.arange(1, len(sorted_err)+1) / len(sorted_err)
    ax.plot(sorted_err, cdf, color=c, linestyle=ls, linewidth=2.5, label=name)

# Reference lines
for thresh_k in [25, 50, 75, 100]:
    ax.axvline(thresh_k * 1000, color='gray', linewidth=0.7, alpha=0.6,
               linestyle=':', label=f'${thresh_k}K' if thresh_k == 50 else '')
    ax.text(thresh_k * 1000 + 1000, 0.02, f'${thresh_k}K', fontsize=8, color='gray')

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.set_xlim(0, 200_000)
ax.set_xlabel('Absolute Prediction Error (USD)', fontsize=LABEL_FS)
ax.set_ylabel('Cumulative % of Test Samples', fontsize=LABEL_FS)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_title('Cumulative Error Distribution — All Models',
             fontsize=TITLE_FS, fontweight='bold')
ax.legend(fontsize=10)

# Annotation
for thresh, name, err in [(50_000, 'RF', err_rf), (50_000, 'GB', err_gb)]:
    pct = (err <= thresh).mean() * 100
    print(f'{name}: {pct:.1f}% of predictions within ${thresh/1e3:.0f}K')

plt.tight_layout()
plt.savefig('../outputs/plot06_10_cumulative_error.png', dpi=FIG_DPI)
plt.show()

---
### 📊 PLOT 11 — Linear Regression Coefficient Analysis
**What it shows:** Signed coefficients of the linear model with 95% confidence intervals (bootstrap).  
**Presentation use:** Only relevant for the linear baseline, but useful for explaining which features push predictions up or down in a simple, interpretable way.

In [ ]:
n_boot = 200
coef_boot = np.zeros((n_boot, len(feature_names)))

rng = np.random.default_rng(42)
for i in range(n_boot):
    idx = rng.choice(len(X_train), size=len(X_train), replace=True)
    lr_b = LinearRegression().fit(X_train[idx], y_train[idx])
    coef_boot[i] = lr_b.coef_

coef_mean = coef_boot.mean(axis=0)
coef_lo   = np.percentile(coef_boot, 2.5,  axis=0)
coef_hi   = np.percentile(coef_boot, 97.5, axis=0)

coef_df = pd.DataFrame({'feature': feature_names, 'mean': coef_mean,
                         'lo': coef_lo, 'hi': coef_hi})
coef_df = coef_df.sort_values('mean')

fig, ax = plt.subplots(figsize=(10, 7))
colors_coef = ['#d62728' if v < 0 else '#2ca02c' for v in coef_df['mean']]
ax.barh(coef_df['feature'], coef_df['mean'],
        xerr=[coef_df['mean']-coef_df['lo'], coef_df['hi']-coef_df['mean']],
        color=colors_coef, edgecolor='white', height=0.6,
        error_kw=dict(ecolor='black', capsize=4, elinewidth=1.2))
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Coefficient (×$100K) ± 95% CI', fontsize=LABEL_FS)
ax.set_title('Linear Regression Coefficients\n(bootstrap 95% confidence intervals)',
             fontsize=TITLE_FS, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/plot06_11_lr_coefficients.png', dpi=FIG_DPI)
plt.show()

---
## Summary of Notebook 06

| Plot | File | Key Takeaway |
|------|------|--------------|
| 1 | plot06_01_pred_vs_actual | GB/RF stick to diagonal; LR under-predicts at high values |
| 2 | plot06_02_residual_distributions | RF/GB residuals narrower and centred; LR has fat tails |
| 3 | plot06_03_residuals_vs_predicted | Moderate heteroscedasticity at high predicted values |
| 4 | plot06_04_geo_error_map | Largest errors in coastal luxury pockets |
| 5 | plot06_05_feature_importance | `median_income` top driver; both models agree |
| 6 | plot06_06_permutation_importance | Confirms income + lat/lon are genuinely predictive |
| 7 | plot06_07_partial_dependence | Income has near-linear effect; latitude shows Bay Area spike |
| 8 | plot06_08_error_by_bracket | Highest absolute error in $450K+ bracket |
| 9 | plot06_09_error_by_proximity | Island districts hardest to predict (few samples) |
| 10 | plot06_10_cumulative_error | ~80% of RF/GB predictions within $50K of truth |
| 11 | plot06_11_lr_coefficients | Income has the largest positive coefficient; bedrooms negative |